In [6]:
import os
from dotenv import find_dotenv, load_dotenv

# Always reload latest values from .env into the current notebook kernel.
ENV_PATH = find_dotenv(filename=".env", usecwd=True)
if not ENV_PATH:
    raise FileNotFoundError("Could not locate a .env file from the current working directory tree.")

load_dotenv(dotenv_path=ENV_PATH, override=True)

True

In [7]:
# OpenAI client setup

from pathlib import Path
from openai import OpenAI
import httpx

cert_env = os.environ.get("CERT_LLM", "").strip()
if not cert_env:
    raise EnvironmentError(
        "CERT_LLM is not set. Point it to your CA bundle file or directory."
    )

cert_path = Path(cert_env).expanduser()
# If CERT_LLM is a directory, use the default bundle filename inside it.
if cert_path.is_dir():
    cert_path = cert_path / "ca-bundle_lite.crt"

SSL_CERT = str(cert_path)
if not cert_path.exists():
    raise FileNotFoundError(f"Certificate bundle not found at: {SSL_CERT}")

BASE_URL = "https://litellm.icp.infineon.com"
MODEL = os.environ.get("OPENAI_MODEL", "gpt-5.2").strip()

http_client = httpx.Client(verify=SSL_CERT)
client = OpenAI(
    base_url=f"{BASE_URL}/v1",
    api_key=os.environ.get("OPENAI_API_KEY"),
    http_client=http_client,
)


In [8]:
model_list = client.models.list()
available_models = [m.id for m in model_list.data if getattr(m, "id", None)]
valid_models = [m for m in available_models if m != "no-default-models"]

if MODEL == "no-default-models":
    MODEL = ""

if MODEL and MODEL not in available_models:
    print(f"Requested model '{MODEL}' is not available for this key.")
    MODEL = valid_models[0] if valid_models else ""

if not MODEL:
    print("No valid model available from the proxy.")
    print("Set OPENAI_MODEL to a model your key can access and rerun this cell.")
    print(f"Models returned by /v1/models: {available_models}")
else:
    print(f"Using model: {MODEL}")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": "this is a test request, write a short poem",
            }
        ],
    )
    

from langchain.chat_models import init_chat_model

# Route through the LiteLLM proxy (reuses the SSL bundle + key); a bare model name
# would default to the public OpenAI endpoint, which the network refuses.
model = init_chat_model(
    MODEL,
    model_provider="openai",
    base_url=f"{BASE_URL}/v1",
    api_key=os.environ.get("OPENAI_API_KEY"),
    http_client=http_client,
)

Using model: gpt-5.2


# Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM. Messages are objects that contain:

- Role: Identifier the message type (eg: system, user)
- Content: It represent the actual content of the message (like: text, image, audio, doc etc)
- Metadata: Optional field such as response information, message IDs and token usage

LangChain provides a standard message type that works across all models providers, ensuring consistent behavior regardless of the model being called.

In [9]:
# Initialize the Model
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3-32b")

#### Text Prompt

Text Prompt are in string format - ideal for straightforward generation tasks where user not needed to retain conversation history.

Example:

In [10]:
# Use simple Human Text Prompt
model.invoke("What is the capital of France?")

AIMessage(content="<think>\nOkay, the user is asking for the capital of France. Let me think. I remember from school that the capital is Paris. But wait, maybe I should double-check to be sure. Let me recall... France is a country in Western Europe, right? Paris is a major city there, known for the Eiffel Tower, the Louvre, and other landmarks. I think that's correct. I don't think there's any confusion with other cities like Lyon or Marseille, which are also big cities but not capitals. The capital of France is definitely Paris. Yeah, that's right. No need to overcomplicate it. Just confirm in my mind that Paris is the capital. Yep, that's the answer.\n</think>\n\nThe capital of France is **Paris**. It is renowned for its historical significance, cultural landmarks such as the Eiffel Tower and the Louvre Museum, and its role as a global city in art, fashion, and gastronomy.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 199, 'prompt_tokens': 15, 'total

The above one provide the simple Text Prompt (Human Text)
And, the model response with AIMessage format.

Use Text Prompt when:
- You have a single, standalone request
- You don't need conversation history
- You want minimal code complexity

### Message Prompt

Alternatively, you can pass in a list of messages to the model by providing a list of message objects.

Message Type:
- System Message: Tells the model how to behave and provide context for interaction
- Human Message: Represents use input and interactions with the model
- AI Message: Response generated by the model, including text content, tool calls and metadata
- Tool Message: Represents the output of tool calls

### System Message

A System Message represent an initial set of instructions that primes the model's behavior. You can use a System Message to set the tone, define the model's role and establish guidelines for responses. 

### Human Message

A Human Message represent user input and interactions. They can contain text, images, audio, file and any other amount of multimodel content.

### AI Message
An AI Message represent the output of the model invocation. They can include Multi-model data, tool calls and provide specific metadata that can access later.

### Tool Message
For models that support tool calling, AI Message can contain Tool Calls. Tool Message are used to pass the result of a single tool execution back to the model.

In [11]:
from langchain.messages import HumanMessage, SystemMessage, AIMessage

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Hello, who won the world series in 2020?")
]

response = model.invoke(messages)
response.content

"<think>\nOkay, the user is asking who won the World Series in 2020. I need to recall the correct team. Let me think. The World Series is the annual championship series of Major League Baseball in North America. The 2020 season was shortened due to the COVID-19 pandemic, so the playoffs were also condensed. \n\nI remember that the Los Angeles Dodgers had a strong team that year. They won the National League West division and went on to the playoffs. In the World Series, they faced the Tampa Bay Rays. The series was a best-of-seven, and the Dodgers won in six games. That would make them the 2020 World Series champions.\n\nWait, let me double-check the dates to make sure I'm not confusing it with another year. The 2020 World Series took place in October 2020, after the regular season was cut to 60 games. The Dodgers did indeed defeat the Rays, winning their first World Series since 1988. \n\nI should also confirm the key players or moments if needed, but the user just asked for the winne

In [14]:
## Details Information on System Messages

system_message = SystemMessage("""
You are a senior Python developer with expertise in web developer framework.
Always provide code examples and explain your reasoning in detail.
Be concise and clear in your explanations, and ensure that your code is well-structured and follows best practices.
""")

messages = [
    system_message,
    HumanMessage(content="Can you provide an example of a simple Flask application?")
]
response = model.invoke(messages)
response.content


'<think>\nOkay, the user is asking for a simple Flask application example. Let me start by recalling what a basic Flask app looks like. I need to make sure it\'s straightforward enough for someone new to Flask.\n\nFirst, I should mention installing Flask. They might not have it installed yet. So I\'ll include the pip install command. Then, the standard "Hello, World!" example is perfect for simplicity. \n\nI\'ll create a Python script, maybe call it app.py. The code will import Flask, create an instance, define a route for the root URL, and return a response. I need to explain each part step by step. \n\nWait, should I include running the app with app.run()? Yes, and maybe mention setting the debug mode, but also a note about not using debug in production. \n\nAlso, maybe add a note about the structure. Like how to run the script using FLASK_APP environment variable or using python directly. Some users might be confused about how to start the server.\n\nLet me check if there\'s anythin

In [16]:
# Message with proper role assignment, Content and Metadata

messages = HumanMessage(
    role="user",
    content="Who will won the FIFA World Cup 2026?",
    name="Alice",
    id="msg-001",
    metadata={"topic": "sports", "language": "en"}
)

response = model.invoke([messages])
response.content

'<think>\nOkay, so the user is asking who will win the FIFA World Cup 2026. Let me start by thinking about the factors that influence such predictions.\n\nFirst, the 2026 World Cup is going to be in the United States, Canada, and Mexico. That\'s a new co-hosting arrangement. Hosting in North America might give teams from there an edge, like the US, Mexico, Canada, maybe even Costa Rica or Panama. But I\'m not sure how big that edge is. Sometimes hosts perform better, but it\'s not always the case.\n\nLooking at the teams, traditionally strong teams are Brazil, Argentina, France, Spain, Germany, Portugal, maybe England. These countries have a history of success and usually have strong squads. Let me check their current form. Brazil has Neymar, who\'s a key player, but he\'s aging. Argentina has Messi, who\'s still at his peak. France has a young squad with Mbappé leading the line. Spain has been dominant in the Euros recently, so their tiki-taka style might still be effective. Germany i